# Chapter 01: k-최근접 이웃(k-NN)을 이용한 이진 분류 분석

## 1. 분석 개요
- **연구 대상**: 도미(Bream) 및 빙어(Smelt) 데이터셋
- **분석 목적**: 생선의 길이와 무게 특성을 바탕으로 종을 분류하는 모델 구축 및 k-NN 알고리즘의 통계적 특성 파악

## 2. 데이터 준비 및 시각화
통계학 전공자로서 모델 피팅 전 데이터의 분포와 특성 간 스케일 차이를 확인하는 EDA를 수행합니다.

In [ ]:
import matplotlib.pyplot as plt
from sklearn.neighbors import KNeighborsClassifier

# 도미 데이터
bream_length = [25.4, 26.3, 26.5, 29.0, 29.0, 29.7, 29.7, 30.0, 30.0, 30.7, 31.0, 31.0, 31.5, 32.0, 32.0, 32.0, 33.0, 33.0, 33.5, 33.5, 34.0, 34.0, 34.5, 35.0, 35.0, 35.0, 35.0, 36.0, 36.0, 37.0, 38.5, 38.5, 39.5, 41.0, 41.0]
bream_weight = [242.0, 290.0, 340.0, 363.0, 430.0, 450.0, 500.0, 390.0, 450.0, 500.0, 475.0, 500.0, 500.0, 340.0, 600.0, 600.0, 700.0, 700.0, 610.0, 650.0, 575.0, 685.0, 620.0, 680.0, 700.0, 725.0, 720.0, 714.0, 850.0, 1000.0, 920.0, 955.0, 925.0, 975.0, 950.0]

# 빙어 데이터
smelt_length = [9.8, 10.5, 10.6, 11.0, 11.2, 11.3, 11.8, 11.8, 12.0, 12.2, 12.4, 13.0, 14.3, 15.0]
smelt_weight = [6.7, 7.5, 7.0, 9.7, 9.8, 8.7, 10.0, 9.9, 9.8, 12.2, 13.4, 12.2, 19.7, 19.9]

plt.figure(figsize=(10, 6))
plt.scatter(bream_length, bream_weight, label='Bream', alpha=0.7)
plt.scatter(smelt_length, smelt_weight, label='Smelt', alpha=0.7)
plt.xlabel('Length (cm)')
plt.ylabel('Weight (g)')
plt.title('Fish Data Distribution')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.5)
plt.show()

print("Analysis: 무게(y축)의 범위가 길이(x축)보다 약 20배 이상 넓음 확인. 스케일 차이에 따른 편향 가능성 존재.")

## 3. 모델 피팅 및 성능 평가
k-NN 알고리즘을 활용하여 분류 모델을 학습시킵니다. 여기서는 Scikit-learn의 표준 워크플로우를 따릅니다.

In [ ]:
length = bream_length + smelt_length
weight = bream_weight + smelt_weight

# 2차원 리스트 구조화
fish_data = [[l, w] for l, w in zip(length, weight)]
fish_target = [1]*35 + [0]*14

kn = KNeighborsClassifier()
kn.fit(fish_data, fish_target)

score = kn.score(fish_data, fish_target)
print(f"Training Set Accuracy: {score}")

## 4. 통계적 실험: 하이퍼파라미터 k의 영향
주변 이웃의 개수 $k$를 변경하며 모델의 결정 경계가 어떻게 변하는지 관찰합니다.

In [ ]:
for k in [5, 15, 30, 49]:
    kn_test = KNeighborsClassifier(n_neighbors=k)
    kn_test.fit(fish_data, fish_target)
    s = kn_test.score(fish_data, fish_target)
    print(f"k={k} -> Accuracy: {s:.4f}")

print("Insight: k가 커질수록 모델은 단순해지며(High Bias), k=49일 때 다수결 원칙에 의해 모든 샘플을 도미로 예측함.")

## 5. 결론 및 비판적 분석
- **일반화 성능 부재**: 현재 1.0의 정확도는 훈련 데이터로 직접 평가한 결과로, **Data Leakage**가 발생함. 실제 연구 환경에서는 반드시 독립적인 테스트 세트 분리가 필요함.
- **특성 스케일링**: 무게와 길이의 단위 차이를 극복하기 위해 차후 **표준 점수(Standard Score)** 도입이 필수적임.
- **비모수 모델**: k-NN은 데이터 분포에 대한 가정이 없는 비모수 모델(Non-parametric model)임을 확인.